In [ ]:
import shutil
import os

if os.path.exists('/content/dataset'):
    shutil.rmtree('/content/dataset')
    print("Cleared the broken dataset folder!")

Cleared the broken dataset folder!


In [ ]:
# =============================================================================
# ACCOUNT 2: ResNet50_Frozen, DenseNet121_Frozen, ResNet50_FT, DenseNet121_FT
# =============================================================================

# --- CELL 1: Setup ---
from google.colab import drive
drive.mount('/content/drive')

import tensorflow as tf
import os, shutil, time, gc
import numpy as np
import pandas as pd
import json, warnings
from collections import Counter

print(f"TensorFlow: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
print(f"GPU: {gpus}")
if not gpus:
    raise RuntimeError("NO GPU! Go to Runtime → Change runtime type → T4 GPU")

# === UPDATED PATHS ===
DATA_ROOT_DRIVE = "/content/drive/MyDrive/Epic and CSCR hospital Dataset_New"
DATA_ROOT = "/content/dataset"
SAVE_DIR = "/content/drive/MyDrive/Brain_Tumor_Results_v2"
os.makedirs(SAVE_DIR, exist_ok=True)

assert os.path.exists(DATA_ROOT_DRIVE), f"Dataset not found at: {DATA_ROOT_DRIVE}"

if not os.path.exists(DATA_ROOT):
    print("Copying dataset to local storage...")
    t0 = time.time()
    shutil.copytree(DATA_ROOT_DRIVE, DATA_ROOT)
    print(f"Done in {time.time()-t0:.0f}s")

print(f"\nDataset structure:")
total_train, total_test = 0, 0
for split in ['Train', 'Test']:
    sp = os.path.join(DATA_ROOT, split)
    for cls in sorted(os.listdir(sp)):
        cp = os.path.join(sp, cls)
        if os.path.isdir(cp):
            n = len([f for f in os.listdir(cp) if f.lower().endswith(('.jpg','.jpeg','.png','.bmp'))])
            print(f"  {split}/{cls}: {n} images")
            if split == 'Train': total_train += n
            else: total_test += n
print(f"\nTotal — Train: {total_train} | Test: {total_test} | All: {total_train + total_test}")

# --- CELL 2: Imports & Pipeline ---
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import ResNet50, DenseNet121
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (f1_score, precision_score, recall_score, accuracy_score)

warnings.filterwarnings('ignore')

IMG_SIZE = (224, 224)
NUM_CLASSES = 4
EPOCHS = 20
N_FOLDS = 5
SEED = 42

CLASS_NAMES = sorted([
    c for c in os.listdir(os.path.join(DATA_ROOT, 'Train'))
    if os.path.isdir(os.path.join(DATA_ROOT, 'Train', c))
])
print(f"Classes: {CLASS_NAMES}")

def load_paths_labels(root, split):
    paths, labels = [], []
    for ci, cn in enumerate(CLASS_NAMES):
        d = os.path.join(root, split, cn)
        if not os.path.isdir(d): continue
        for f in sorted(os.listdir(d)):
            if f.lower().endswith(('.jpg','.jpeg','.png','.bmp')):
                paths.append(os.path.join(d, f))
                labels.append(ci)
    return np.array(paths), np.array(labels)

all_train_paths, all_train_labels = load_paths_labels(DATA_ROOT, 'Train')
test_paths, test_labels = load_paths_labels(DATA_ROOT, 'Test')
print(f"Train: {len(all_train_paths)} | Test: {len(test_paths)}")

def create_dataset(paths, labels, batch_size=32, augment=False, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(len(paths), seed=SEED)
    def _parse(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_image(img, channels=3, expand_animations=False)
        img = tf.image.resize(img, IMG_SIZE)
        img = tf.cast(img, tf.float32) / 255.0
        return img, tf.one_hot(label, NUM_CLASSES)
    def _augment(img, label):
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_flip_up_down(img)
        img = tf.image.random_brightness(img, 0.15)
        img = tf.image.random_contrast(img, 0.85, 1.15)
        img = tf.image.rot90(img, k=tf.random.uniform([], 0, 4, dtype=tf.int32))
        return tf.clip_by_value(img, 0.0, 1.0), label
    ds = ds.map(_parse, num_parallel_calls=tf.data.AUTOTUNE)
    if augment:
        ds = ds.map(_augment, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

test_dataset = create_dataset(test_paths, test_labels, augment=False, shuffle=False)

# --- CELL 3: Model Definitions (pretrained only) ---
def _build_pretrained(base_cls, preproc_fn, name, finetune_layers=0):
    base = base_cls(weights='imagenet', include_top=False, input_shape=(224,224,3))
    if finetune_layers == 0:
        base.trainable = False
    else:
        base.trainable = True
        for layer in base.layers[:-finetune_layers]:
            layer.trainable = False
    inp = layers.Input(shape=(224,224,3))
    x = layers.Lambda(lambda img: preproc_fn(img * 255.0))(inp)
    x = base(x, training=(finetune_layers > 0))
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(128, activation='relu', name='feat128')(x)
    x = layers.Dropout(0.3)(x)
    return Model(inp, layers.Dense(4, activation='softmax')(x), name=name)

def build_resnet50_frozen():
    return _build_pretrained(ResNet50, tf.keras.applications.resnet50.preprocess_input,
                             'ResNet50_Frozen', 0)
def build_densenet121_frozen():
    return _build_pretrained(DenseNet121, tf.keras.applications.densenet.preprocess_input,
                             'DenseNet121_Frozen', 0)
def build_resnet50_ft():
    return _build_pretrained(ResNet50, tf.keras.applications.resnet50.preprocess_input,
                             'ResNet50_FineTuned', 20)
def build_densenet121_ft():
    return _build_pretrained(DenseNet121, tf.keras.applications.densenet.preprocess_input,
                             'DenseNet121_FineTuned', 30)

MODEL_CONFIGS = {
    'ResNet50_Frozen':       {'build_fn': build_resnet50_frozen,    'lr': 1e-3, 'bs': 32},
    'DenseNet121_Frozen':    {'build_fn': build_densenet121_frozen, 'lr': 1e-3, 'bs': 32},
    'ResNet50_FineTuned':    {'build_fn': build_resnet50_ft,        'lr': 1e-4, 'bs': 32},
    'DenseNet121_FineTuned': {'build_fn': build_densenet121_ft,     'lr': 1e-4, 'bs': 32},
}

print(f"Account 2 models: {list(MODEL_CONFIGS.keys())}")

# --- CELL 4: 5-Fold CV ---
CHECKPOINT_PATH = os.path.join(SAVE_DIR, 'cv_account2.json')
all_results = {}

if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH, 'r') as f:
        all_results = json.load(f)
    print(f"Loaded checkpoint: {list(all_results.keys())}")

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

for model_name, cfg in MODEL_CONFIGS.items():
    if model_name in all_results:
        print(f"SKIPPING {model_name} (already done)")
        continue

    print(f"\n{'='*70}")
    print(f"Training: {model_name} ({N_FOLDS}-Fold CV)")
    print(f"{'='*70}")

    fold_val_metrics, fold_test_preds, fold_test_probs = [], [], []
    fold_val_accuracies, fold_times = [], []

    for fi, (tr_idx, va_idx) in enumerate(skf.split(all_train_paths, all_train_labels)):
        print(f"  Fold {fi+1}/{N_FOLDS}", end=" ... ", flush=True)
        train_ds = create_dataset(all_train_paths[tr_idx], all_train_labels[tr_idx],
                                  batch_size=cfg['bs'], augment=True)
        val_ds = create_dataset(all_train_paths[va_idx], all_train_labels[va_idx],
                                batch_size=cfg['bs'], augment=False, shuffle=False)
        tf.keras.backend.clear_session()
        gc.collect()

        model = cfg['build_fn']()
        model.compile(optimizer=keras.optimizers.Adam(learning_rate=cfg['lr']),
                      loss='categorical_crossentropy', metrics=['accuracy'])

        t0 = time.time()
        hist = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS,
            callbacks=[
                EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=0),
                ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=0)
            ], verbose=0)
        dt = time.time() - t0
        fold_times.append(dt)

        val_probs, val_true = [], []
        for bi, bl in val_ds:
            val_probs.extend(model.predict(bi, verbose=0))
            val_true.extend(np.argmax(bl.numpy(), axis=1))
        val_probs, val_true = np.array(val_probs), np.array(val_true)
        val_pred = np.argmax(val_probs, axis=1)

        va_acc = accuracy_score(val_true, val_pred)
        va_f1 = f1_score(val_true, val_pred, average='weighted')
        va_prec = precision_score(val_true, val_pred, average='weighted')
        va_rec = recall_score(val_true, val_pred, average='weighted')

        fold_val_metrics.append({'accuracy': va_acc, 'f1_score': va_f1,
                                 'precision': va_prec, 'recall': va_rec})
        fold_val_accuracies.append(va_acc)

        test_probs, test_true = [], []
        for bi, bl in test_dataset:
            test_probs.extend(model.predict(bi, verbose=0))
            test_true.extend(np.argmax(bl.numpy(), axis=1))
        test_probs, test_true = np.array(test_probs), np.array(test_true)

        fold_test_preds.append(np.argmax(test_probs, axis=1).tolist())
        fold_test_probs.append(test_probs.tolist())

        test_acc = accuracy_score(test_true, np.argmax(test_probs, axis=1))
        n_epochs = len(hist.history['loss'])
        print(f"Val={va_acc:.4f} F1={va_f1:.4f} | Test={test_acc:.4f} | Ep={n_epochs} | {dt:.0f}s")

        del model, train_ds, val_ds, hist, val_probs, val_true, val_pred, test_probs
        tf.keras.backend.clear_session()
        gc.collect()

    fdf = pd.DataFrame(fold_val_metrics)
    best_fold = int(np.argmax(fold_val_accuracies))

    all_results[model_name] = {
        'fold_val_metrics': fold_val_metrics,
        'fold_val_accuracies': fold_val_accuracies,
        'test_predictions': fold_test_preds,
        'test_probabilities': fold_test_probs,
        'test_true': test_true.tolist(),
        'best_fold': best_fold,
        'fold_times': fold_times,
        'cv_accuracy_mean': float(fdf['accuracy'].mean()),
        'cv_accuracy_std': float(fdf['accuracy'].std()),
        'cv_f1_mean': float(fdf['f1_score'].mean()),
        'cv_f1_std': float(fdf['f1_score'].std()),
        'cv_precision_mean': float(fdf['precision'].mean()),
        'cv_precision_std': float(fdf['precision'].std()),
        'cv_recall_mean': float(fdf['recall'].mean()),
        'cv_recall_std': float(fdf['recall'].std()),
        'mean_time': float(np.mean(fold_times)),
        'n_folds': N_FOLDS,
    }

    print(f"\n  {model_name}: CV={fdf['accuracy'].mean():.4f}+/-{fdf['accuracy'].std():.4f} | Best Fold={best_fold+1}")

    with open(CHECKPOINT_PATH, 'w') as f:
        json.dump(all_results, f, indent=2)
    print(f"  Saved to {CHECKPOINT_PATH}")
    gc.collect()

print(f"\n{'='*70}")
print(f"ACCOUNT 2 COMPLETE: {list(all_results.keys())}")
print(f"{'='*70}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
TensorFlow: 2.19.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Copying dataset to local storage...
Done in 207s

Dataset structure:
  Train/glioma: 3018 images
  Train/meningioma: 2183 images
  Train/notumor: 1945 images
  Train/pituitary: 2150 images
  Test/glioma: 603 images
  Test/meningioma: 436 images
  Test/notumor: 389 images
  Test/pituitary: 429 images

Total — Train: 9296 | Test: 1857 | All: 11153
Classes: ['glioma', 'meningioma', 'notumor', 'pituitary']
Train: 9296 | Test: 1857
Account 2 models: ['ResNet50_Frozen', 'DenseNet121_Frozen', 'ResNet50_FineTuned', 'DenseNet121_FineTuned']

Training: ResNet50_Frozen (5-Fold CV)
  Fold 1/5 ... Downloading data from https://storage.googleapis.com/tensorflow/keras-applications/resnet/resnet50_weights_tf_dim_ordering_tf_kernels_notop.h5
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us